# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [ ]:
print("Hello World!")


In [ ]:
import pandas as pd
import pm4py

print("works")

In [ ]:
import pm4py

log = pm4py.read_xes("../data/BPI Challenge 2017.xes.gz")

print(log)

In [ ]:
df = pm4py.convert_to_dataframe(log)

df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
len(df)

In [ ]:
# Number of Cases
df["case:concept:name"].nunique()

In [ ]:
# Most common activities
df["concept:name"].value_counts().head(20)

In [ ]:
dfg, start_activities, end_activities = pm4py.discover_dfg(log)

pm4py.view_dfg(
    dfg,
    start_activities,
    end_activities
)

In [ ]:
print("Hello")

# Milestone 1

In [ ]:
import pandas as pd
import pm4py
import numpy as np
import random

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("works")

In [ ]:
#import pm4py

#log = pm4py.read_xes("../data/BPI Challenge 2017.xes.gz")

#print(log)

In [ ]:
#df = pm4py.convert_to_dataframe(log)

#df.head()

## Polluter Script

### Pattern: Polluted Labels

In [ ]:
df.head()

In [ ]:
def pollute_labels(
    df,
    rate=0.5,
    activity_column="concept:name",
    case_column="case:concept:name",
    pollution_column="pollution_type"
):
    df = df.copy()

    if pollution_column not in df.columns:
        df[pollution_column] = None

    n = int(len(df) * rate)
    indices = np.random.choice(df.index, n, replace=False)

    for index in indices:
        activity = df.loc[index, activity_column]
        case_id = df.loc[index, case_column]

        df.loc[index, activity_column] = f"{activity} - Incident No. {case_id}"
        df.loc[index, pollution_column] = "polluted_label"

    return df

In [ ]:
#df_polluted = pollute_labels(df)
#df_polluted.head(100)

### Pattern: Distorted Labels

In [ ]:
df["concept:name"].unique()

In [ ]:
df["concept:name"].value_counts()

In [ ]:
def distort_activity_label(label):
    replacements = {
        "application": "appl.",
        "Application": "App.",
        "offers": "ofrs.",
        "Offer": "Off.",
        "incomplete": "incompl.",
        "Incomplete": "Incompl.",
        "files": "docs.",
        "leads": "lds.",
        "Handle": "Hndl.",
        "Validate": "Valid.",
        "Validating": "Valid.",
        "Complete": "Comp.",
        "Created": "Crtd.",
        "Create": "Crt.",
        "Accepted": "Acc.",
        "Cancelled": "Canc.",
        "Submitted": "Subm.",
        "Pending": "Pend.",
        "Denied": "Den.",
        "Refused": "Ref.",
        "Returned": "Ret.",
        "potential": "pot.",
        "fraud": "frd.",
        "Shortened": "Short.",
        "completion": "compl.",
        "Personal": "Pers.",
        "Loan": "Ln.",
        "collection": "coll.",
        "online": "onl.",
        "mail": "email"
    }

    distorted = str(label)

    for original, replacement in replacements.items():
        if original in distorted:
            return distorted.replace(original, replacement, 1)

    return distorted + "."

In [ ]:
labels = df["concept:name"].dropna().unique()

test_df = pd.DataFrame({
    "original_label": labels,
    "distorted_label": [distort_activity_label(label) for label in labels]
})

test_df

In [ ]:
def pollute_distorted_labels(
    df,
    activity_column="concept:name",
    rate=0.05,
    pollution_column="pollution_type"
):
    df = df.copy()

    if pollution_column not in df.columns:
        df[pollution_column] = None

    n = int(len(df) * rate)
    indices = np.random.choice(df.index, n, replace=False)

    for index in indices:
        original_label = df.loc[index, activity_column]

        if pd.notna(original_label):
            distorted_label = distort_activity_label(original_label)

            df.loc[index, activity_column] = distorted_label
            df.loc[index, pollution_column] = "distorted_label"

    return df

In [ ]:
# df_polluted = pollute_distorted_labels(
#     df_polluted,
#     activity_column="concept:name",
#     rate=0.05
# )

In [ ]:
# print("Original unique activities:", df["concept:name"].nunique())
# print("Polluted unique activities:", df_polluted["concept:name"].nunique())
#
# df_polluted["pollution_type"].value_counts()

In [ ]:
# df_polluted["concept:name"].value_counts()

In [ ]:
df_polluted = df.copy()

df_polluted = pollute_labels(
    df_polluted,
    rate=0.5
)

In [ ]:
df_polluted.head(100)

In [ ]:
df_polluted = pollute_distorted_labels(
    df_polluted,
    activity_column="concept:name",
    rate=0.05
)

In [ ]:
pm4py.write_xes(df_polluted, "polluted_log.xes")

In [ ]:
log_polluted = pm4py.read_xes("polluted_log.xes")

In [ ]:
dfg_polluted, start_activities_polluted, end_activities_polluted = pm4py.discover_dfg(log_polluted)

pm4py.view_dfg(
    dfg_polluted,
    start_activities_polluted,
    end_activities_polluted
)

In [ ]:
# Anzahl Cases
cases_polluted = df_polluted["case:concept:name"].nunique()
# Anzahl Cases
cases = df["case:concept:name"].nunique()
print(f"Anzahl Cases: {cases}")
print(f"Anzahl Cases Polluted: {cases_polluted}")

# Anzahl verschiedener Aktivitäten
distinct_activities = df["concept:name"].nunique()
# Anzahl verschiedener Aktivitäten
distinct_activities_polluted = df_polluted["concept:name"].nunique()
print(f"Distinct Activities: {distinct_activities}")
print(f"Distinct Activities Polluted: {distinct_activities_polluted}")

# Anzahl verschiedener Traces / Varianten
variants = pm4py.get_variants_as_tuples(df)
traces = len(variants)

variants_polluted = pm4py.get_variants_as_tuples(df_polluted)
traces_polluted = len(variants_polluted)

print(f"Variants: {traces}")
print(f"Variants Polluted: {traces_polluted}")

## Cleaner Script

In [ ]:
df_polluted.head(100)

In [ ]:
def clean_polluted_labels(
    df,
    activity_column="concept:name",
    cleaning_column="label_cleaning"
):
    df = df.copy()

    if cleaning_column not in df.columns:
        df[cleaning_column] = None

    original_labels = df[activity_column].copy()

    # alles als string behandeln
    df[activity_column] = df[activity_column].astype("string")

    patterns = [
        r"\s*-\s*Incident No\.?\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Incident\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Case\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Application\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*User\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Resource\s*[A-Za-z0-9_\-]+",
        r"\s*#\s*[A-Za-z0-9_\-]+",
        r"\s*\(\s*id\s*[:=]?\s*[A-Za-z0-9_\-]+\s*\)",
        r"\s*\(\s*case\s*[:=]?\s*[A-Za-z0-9_\-]+\s*\)",
    ]

    for pattern in patterns:
        df[activity_column] = df[activity_column].str.replace(
            pattern,
            "",
            regex=True
        )

    # Whitespace normalisieren
    df[activity_column] = (
        df[activity_column]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    changed_mask = original_labels.astype("string") != df[activity_column]

    df.loc[changed_mask, cleaning_column] = "cleaned_polluted_label"

    return df

In [ ]:
df_cleaned = df_polluted.copy()
df_cleaned = clean_polluted_labels(
    df_cleaned,
    activity_column="concept:name"
)
df_cleaned.head(100)